# 🎧🎬 تبدیل صدا و ویدیو (فارسی) به متن — حالتِ دسته‌ای (Batch)

این نوت‌بوک **همه‌ی فایل‌های صوتی و ویدیوییِ یک پوشه** را یک‌جا به متن تبدیل می‌کند (Whisper large-v3).
برای ویدیو لازم نیست کاری کنی — صدای داخلِ ویدیو خودکار استخراج می‌شود.
هر فایل که قبلاً انجام شده باشد رد می‌شود؛ اگر Colab وسطِ کار قطع شد، فقط دوباره اجرا کن.

### قبل از شروع: **Runtime → Change runtime type → T4 GPU → Save**
بعد سلول‌ها را به‌ترتیب اجرا کن (▶).


## ۱) نصب و بررسی GPU


In [ ]:
!pip -q install faster-whisper
import torch
print("GPU روشن است؟ ", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("⚠️ False بود؟ Runtime → Change runtime type → T4 GPU، بعد این سلول را دوباره اجرا کن.")


## ۲) وصل‌کردن Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## ۳) مسیرِ پوشه را بده و «همه» (صوت + ویدیو) را متن کن
از پنلِ چپ 📁 پوشه‌ای که فایل‌های صوتی/ویدیویی داخلش است را پیدا کن، راست‌کلیک → **Copy path** و در `MEDIA_DIR` بچسبان.
خروجی‌ها در زیرپوشه‌ی `transcripts` ذخیره می‌شود.


In [ ]:
# ↓↓↓ فقط این خط را تنظیم کن ↓↓↓
MEDIA_DIR = "/content/drive/MyDrive/نام-پوشه"   # ← مسیرِ پوشه‌ی فایل‌های صوتی/ویدیویی
# ↑↑↑ ----------------------- ↑↑↑

import os, glob, torch
from faster_whisper import WhisperModel

assert os.path.isdir(MEDIA_DIR), f"پوشه پیدا نشد: {MEDIA_DIR}"
OUTPUT_DIR = os.path.join(MEDIA_DIR, "transcripts")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# صوت + ویدیو (Whisper صدای ویدیو را خودکار استخراج می‌کند)
EXTS = (".m4a",".mp3",".wav",".ogg",".opus",".aac",".flac",".wma",  # audio
        ".mp4",".mov",".mkv",".avi",".webm",".m4v",".flv",".3gp",".mpeg",".mpg")  # video
files = sorted(f for f in glob.glob(os.path.join(MEDIA_DIR,'*')) if f.lower().endswith(EXTS))
print(f"{len(files)} فایل صوتی/ویدیویی پیدا شد.")

device = "cuda" if torch.cuda.is_available() else "cpu"
compute_type = "float16" if device=="cuda" else "int8"
print(f"بارگذاری مدل large-v3 روی {device} ... (بار اول ~۳ گیگ دانلود می‌شود)")
model = WhisperModel("large-v3", device=device, compute_type=compute_type)

def hms(s):
    h=int(s//3600); m=int((s%3600)//60); sec=int(s%60)
    return f"{h:02d}:{m:02d}:{sec:02d}"

for i, path in enumerate(files, 1):
    base = os.path.splitext(os.path.basename(path))[0]
    txt_path = os.path.join(OUTPUT_DIR, base + ".txt")
    ts_path  = os.path.join(OUTPUT_DIR, base + "_timestamps.txt")
    if os.path.exists(txt_path):
        print(f"[{i}/{len(files)}] ⏭  قبلاً انجام شده: {base}")
        continue
    print(f"\n[{i}/{len(files)}] ▶ شروع: {base}")
    segments, info = model.transcribe(path, language="fa", beam_size=5, vad_filter=True,
                                      vad_parameters=dict(min_silence_duration_ms=500))
    total = info.duration or 0
    with open(txt_path,"w",encoding="utf-8") as f1, open(ts_path,"w",encoding="utf-8") as f2:
        for seg in segments:
            line = seg.text.strip()
            f1.write(line+"\n")
            f2.write(f"[{hms(seg.start)} → {hms(seg.end)}] {line}\n")
            pct = (seg.end/total*100) if total else 0
            print(f"\r   {pct:5.1f}%  تا {hms(seg.end)}", end="")
    print(f"\n   ✅ ذخیره شد: {txt_path}")

print("\n\n🎉 همه‌ی فایل‌ها تمام شد! خروجی‌ها در پوشه‌ی transcripts است.")


## ۴) لیست خروجی‌ها


In [ ]:
import glob, os
for p in sorted(glob.glob(os.path.join(OUTPUT_DIR,"*.txt"))):
    kb = os.path.getsize(p)//1024
    print(f"{kb:6d} KB  {os.path.basename(p)}")


## تمام!
فایل‌های متنی در پوشه‌ی **`transcripts`** ذخیره شده‌اند. کافیه بگی «آماده‌ست» تا بخوانمشان و به سیستم اضافه کنم. 🎯
